# DATA 2001 Group Assignment 2026

**Topic:** NSW regional statistics, Greater Sydney POI data, SA2 resource scoring, and report analysis.

**Due:** 20 May 2026, 11:59 PM

This notebook is structured to document the full workflow required for the group submission.

## Deliverables Checklist

- PDF report
- Jupyter Notebook describing the full workflow
- Tutor conversation in Week 12 or Week 13
- One group ZIP file submitted to Canvas

## Setup

In [ ]:
from pathlib import Path
from tabulate import tabulate
import json
import math
import sqlite3
import urllib.parse
import urllib.request

import numpy as np
import pandas as pd
import geopandas as gpd
import requests

DATA_DIR = Path.cwd()
CSV_PATH = DATA_DIR / "Region summary_ New South Wales STE 1.csv"
DB_PATH = DATA_DIR / "data2001_assignment.sqlite"

CSV_PATH

# Task 1: NSW Summary Statistics

## 1.1 Load the NSW Region Summary CSV

In [ ]:
df_raw = pd.read_csv(CSV_PATH)
df_raw.head()

In [ ]:
print("Shape:", df_raw.shape)
display(df_raw.info())
display(df_raw.describe(include="all"))

## 1.2 Data Cleaning

In [ ]:
df_clean = df_raw.copy()

# Standardise column names and text fields.
df_clean.columns = df_clean.columns.str.strip()
text_columns = ["Measure Code", "Parent Description", "Description"]
for column in text_columns:
    df_clean[column] = df_clean[column].astype("string").str.strip()

year_columns = [column for column in df_clean.columns if column.isdigit()]
df_clean[year_columns] = df_clean[year_columns].apply(pd.to_numeric, errors="coerce")

# Remove exact duplicate rows if present.
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

print("Cleaned shape:", df_clean.shape)
df_clean.head()

In [ ]:
missing_summary = (
    df_clean.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)
missing_summary["missing_percent"] = (missing_summary["missing_count"] / len(df_clean) * 100).round(2)
missing_summary

### Cleaning Notes

- The identifier columns are `Measure Code`, `Parent Description`, and `Description`.
- Year columns from 2011 to 2025 were detected and converted to numeric values.
- Missing values appear because not every statistic is available for every year. For example, 2025 only has limited available data, so most time-based analysis uses 2019-2024 instead.
- Exact duplicate rows were checked and removed if present.
- Text columns were stripped of extra whitespace to make filtering and matching more reliable.
- Numeric year columns were converted with `pd.to_numeric(errors="coerce")`, so invalid values become missing values rather than breaking the workflow.


## 1.3 Derived Statistics

Each group member should contribute 5 derived statistics. Use this section to clearly label each member's work.

In [ ]:
latest_year = max(year_columns, key=int)
usable_years = [column for column in year_columns if df_clean[column].notna().any()]
latest_usable_year = max(usable_years, key=int)

print("All year columns:", year_columns)
print("Latest year column:", latest_year)
print("Latest usable year:", latest_usable_year)

### Ronnie Luo: Derived Statistics

Planned statistics:

1. Child share of population: percentage of NSW residents aged 0-14.
2. Older population share: percentage of NSW residents aged 65 and over.
3. Ageing index: number of residents aged 65+ per 100 children aged 0-14.
4. Age dependency ratio: children and older residents per 100 working-age residents.
5. Population density change: change in persons per square kilometre from 2019 to 2024.

In [ ]:
analysis_years = [str(year) for year in range(2019, 2025)]

def metric_series(description):
    matches = df_clean[df_clean["Description"].eq(description)]
    if matches.empty:
        raise ValueError(f"Metric not found: {description}")
    return matches.iloc[0][analysis_years].astype(float)

def age_group_total(age_labels):
    total = pd.Series(0.0, index=analysis_years)
    for sex in ["Males", "Females"]:
        for age_label in age_labels:
            total += metric_series(f"{sex} - {age_label} (no.)")
    return total

total_population = metric_series("Estimated resident population (no.)")
working_age_population = metric_series("Working age population (aged 15-64 years) (no.)")
working_age_share = metric_series("Working age population (aged 15-64 years) (%)")
population_density = metric_series("Population density (persons/km2)")

children_0_14 = age_group_total(["0-4 years", "5-9 years", "10-14 years"])
older_65_plus = age_group_total(["65-69 years", "70-74 years", "75-79 years", "80-84 years", "85 and over"])

ronnie_stats = pd.DataFrame({
    "year": analysis_years,
    "child_share_pct": (children_0_14 / total_population * 100).round(2).values,
    "older_share_pct": (older_65_plus / total_population * 100).round(2).values,
    "ageing_index_older_per_100_children": (older_65_plus / children_0_14 * 100).round(2).values,
    "dependency_ratio_dependents_per_100_working_age": ((children_0_14 + older_65_plus) / working_age_population * 100).round(2).values,
    "working_age_share_pct": working_age_share.round(2).values,
    "population_density_persons_per_km2": population_density.round(2).values,
})

ronnie_summary = pd.DataFrame({
    "derived_statistic": [
        "Child share of population",
        "Older population share",
        "Ageing index",
        "Age dependency ratio",
        "Population density",
    ],
    "2019_value": [
        ronnie_stats.loc[0, "child_share_pct"],
        ronnie_stats.loc[0, "older_share_pct"],
        ronnie_stats.loc[0, "ageing_index_older_per_100_children"],
        ronnie_stats.loc[0, "dependency_ratio_dependents_per_100_working_age"],
        ronnie_stats.loc[0, "population_density_persons_per_km2"],
    ],
    "2024_value": [
        ronnie_stats.loc[5, "child_share_pct"],
        ronnie_stats.loc[5, "older_share_pct"],
        ronnie_stats.loc[5, "ageing_index_older_per_100_children"],
        ronnie_stats.loc[5, "dependency_ratio_dependents_per_100_working_age"],
        ronnie_stats.loc[5, "population_density_persons_per_km2"],
    ],
})
ronnie_summary["change"] = (ronnie_summary["2024_value"] - ronnie_summary["2019_value"]).round(2)
ronnie_summary["interpretation"] = [
    "Percentage-point change in residents aged 0-14.",
    "Percentage-point change in residents aged 65+.",
    "Change in older residents per 100 children.",
    "Change in dependents per 100 working-age residents.",
    "Change in persons per square kilometre.",
]

display(ronnie_stats)
display(ronnie_summary)

### Spencer: Derived Statistics

Planned statistics:

1. Business Entry and Exit Rate
2. Employee Income Mean/Median Ratio
3. Share of Business Sizes
4. Capital Gains Mean/Median Ratio
5. Wage Growth Rate

In [ ]:
rows = []

for i in range(2021, 2025):
    totals = df_clean[df_clean['Description'] == 'Total number of businesses'][str(i)].values[0]
    exits = df_clean[df_clean['Description'] == 'Total number of business exits'][str(i)].values[0]
    entries = df_clean[df_clean['Description'] == 'Total number of business entries'][str(i)].values[0]

    exit_rate = (exits / totals) * 100
    entry_rate = (entries / totals) * 100

    rows.append([i, f"{int(totals)}", f"{int(exits)}", f"{round(exit_rate, 1)}%", f"{int(entries)}", f"{round(entry_rate, 1)}%"])


print(tabulate(rows, headers=["Year", "Total Businesses", "Exits", "Exit Rate", "Entries", "Entry Rate"], tablefmt="pretty"))

In [ ]:
rows = []

for i in range(2018, 2023):
    median = df_clean[df_clean['Description'] == 'Median employee income ($)'][str(i)].values[0]
    mean = df_clean[df_clean['Description'] == 'Mean employee income ($)'][str(i)].values[0]
    
    ratio = mean / median
    rows.append([i, f"${int(mean)}", f"${int(median)}", f"{round(ratio, 2)}"])

print(tabulate(rows, headers=["Year", "Mean Employee Income ($)", "Median Employee Income ($)", "Mean-Median Ratio"], tablefmt="pretty"))

In [ ]:
rows = []

for i in range(2020, 2025):
    non_employing = df_clean[df_clean['Description'] == 'Number of non-employing businesses'][str(i)].values[0]
    small = df_clean[df_clean['Description'] == 'Number of employing businesses: 1-4 employees'][str(i)].values[0]
    medium = df_clean[df_clean['Description'] == 'Number of employing businesses: 5-19 employees'][str(i)].values[0]
    large = df_clean[df_clean['Description'] == 'Number of employing businesses: 20 or more employees'][str(i)].values[0]

    count = non_employing + small + medium + large

    rows.append([i, f"{round(non_employing/count * 100, 1)}%", f"{round(small/count * 100, 1)}%", f"{round(medium/count * 100, 1)}%", f"{round(100 - round(non_employing/count * 100, 1) - round(small/count * 100, 1) - round(medium/count * 100, 1), 1)}%"])
    
print(tabulate(rows, headers=["Year", "Share of Non-Employing", "Share of Micro (1-4 staff)", "Share of Small (5-19 staff)", "Share of Large (20+ staff)"], tablefmt="pretty"))

In [ ]:
rows = []

for i in range(2017, 2023):
    claimants = int(df_clean[df_clean['Description'] == 'Persons who reported gross capital gains (no.)'][str(i)].values[0])
    median_cg = df_clean[df_clean['Description'] == 'Median value of gross capital gains ($)'][str(i)].values[0]
    mean_cg = df_clean[df_clean['Description'] == 'Mean value of gross capital gains ($)'][str(i)].values[0]

    ratio = mean_cg / median_cg
    rows.append([str(i), str(claimants), f"{int(mean_cg)}", f"{int(median_cg)}", f"{round(ratio, 1)}"])

print(tabulate(rows, headers=["Year", "Claimants", "Mean Capital Gain ($)", "Median Capital Gain ($)", "Mean/Median Ratio"], tablefmt="pretty"))

In [ ]:
rows = []

years = [2018, 2019, 2020, 2021, 2022]

for i in range(1, len(years)):
    prev = df_clean[df_clean['Description'] == 'Median employee income ($)'][str(years[i-1])].values[0]
    curr = df_clean[df_clean['Description'] == 'Median employee income ($)'][str(years[i])].values[0]

    growth = ((curr - prev) / prev) * 100
    rows.append([f"{years[i-1]}-{years[i]}", f"${int(prev):,}", f"${int(curr):,}", f"{round(growth, 1)}%"])

print(tabulate(rows, headers=["Period", "Previous Median ($)", "Current Median ($)", "Wage Growth Rate"], tablefmt="pretty"))

### Arya: Derived Statistics


Planned statistics:

1. Median age of persons change (2019 to 2024)
2. Working-age population growth (2019 to 2024)
3. Working-age share change (2019 to 2024)
4. Female median age minus overall median age in 2024
5. Male median age minus overall median age in 2024

In [ ]:
median_age_2019 = df_clean.loc[df_clean["Description"] == "Median age - persons (years)", "2019"].values[0]
median_age_2024 = df_clean.loc[df_clean["Description"] == "Median age - persons (years)","2024"].values[0]

median_age_change = median_age_2024 - median_age_2019

print(f"Median age in 2019: {median_age_2019:.1f} years")
print(f"Median age in 2024: {median_age_2024:.1f} years")
print(f"Change in median age from 2019 to 2024: {median_age_change:.2f} years")

In [ ]:
working_age_2019 = df_clean.loc[df_clean["Description"] == "Working age population (aged 15-64 years) (no.)", "2019"].values[0]
working_age_2024 = df_clean.loc[df_clean["Description"] == "Working age population (aged 15-64 years) (no.)", "2024"].values[0]
working_age_growth_pct = ((working_age_2024 - working_age_2019) / working_age_2019) * 100

print(f"Working-age population in 2019: {working_age_2019:,.0f}")
print(f"Working-age population in 2024: {working_age_2024:,.0f}")
print(f"Working-age population growth from 2019 to 2024: {working_age_growth_pct:.2f}%")

In [ ]:
working_share_2019 = df_clean.loc[df_clean["Description"] == "Working age population (aged 15-64 years) (%)", "2019"].values[0]
working_share_2024 = df_clean.loc[df_clean["Description"] == "Working age population (aged 15-64 years) (%)", "2024"].values[0]
working_share_change = working_share_2024 - working_share_2019

print(f"Working-age share in 2019: {working_share_2019:.1f}%")
print(f"Working-age share in 2024: {working_share_2024:.1f}%")
print(f"Change in working-age share from 2019 to 2024: {working_share_change:.2f} percentage points")

In [ ]:
female_median_age_2024 = df_clean.loc[df_clean["Description"] == "Median age - females (years)", "2024"].values[0]
overall_median_age_2024 = df_clean.loc[df_clean["Description"] == "Median age - persons (years)", "2024"].values[0]
female_vs_overall_gap = female_median_age_2024 - overall_median_age_2024

print(f"Female median age in 2024: {female_median_age_2024:.1f} years")
print(f"Overall median age in 2024: {overall_median_age_2024:.1f} years")
print(f"Female minus overall median age in 2024: {female_vs_overall_gap:.2f} years")

In [ ]:
male_median_age_2024 = df_clean.loc[df_clean["Description"] == "Median age - males (years)", "2024"].values[0]
overall_median_age_2024 = df_clean.loc[df_clean["Description"] == "Median age - persons (years)", "2024"].values[0]
male_vs_overall_gap = male_median_age_2024 - overall_median_age_2024

print(f"Male median age in 2024: {male_median_age_2024:.1f} years")
print(f"Overall median age in 2024: {overall_median_age_2024:.1f} years")
print(f"Male minus overall median age in 2024: {male_vs_overall_gap:.2f} years")

### Ananya: Derived Statistics


Planned statistics:

1. Statistic 1: Population growth rate (2019 to 2024): NSW's average annual population growth rate over the most data-rich 5-year window in the dataset.
2. Statistic 2: Gender balance ratio over time. Tracks the male-to-female population ratio across each year, revealing demographic shifts over the decade.
3. Statistic 3: Year with the most data coverage bonus candidate. Finds which year has the most non-null values across all 800 measure
4. Statistic 4: Median age gap between males and females bonus candidate. Computes the difference in median age between males and females each year. 
5. Statistic 5:  Breadth of topic coverage across parent categories bonus candidate

In [ ]:
pop_2019 = df_clean.loc[df_clean["Description"] == "Estimated resident population (no.)", "2019"].values[0]
pop_2024 = df_clean.loc[df_clean["Description"] == "Estimated resident population (no.)", "2024"].values[0]

cagr = ((pop_2024 / pop_2019) ** (1/5) - 1) * 100

print(f"NSW population 2019: {pop_2019:,.0f}")
print(f"NSW population 2024: {pop_2024:,.0f}")
print(f"Average annual growth rate (CAGR): {cagr:.2f}%")

In [ ]:
males = df_clean[df_clean["Description"].str.contains("males \(no\.\)", na=False, regex=True)]
females = df_clean[df_clean["Description"].str.contains("females \(no\.\)", na=False, regex=True)]

year_cols = [c for c in df_clean.columns if c.isdigit()]

gender_ratio = pd.DataFrame({
    "year": year_cols,
    "males": males[year_cols].values[0],
    "females": females[year_cols].values[0]
}).dropna()

gender_ratio["male_to_female_ratio"] = (gender_ratio["males"] / gender_ratio["females"]).round(4)

print(gender_ratio[["year", "male_to_female_ratio"]].to_string(index=False))
print(f"\nMost balanced year: {gender_ratio.loc[gender_ratio['male_to_female_ratio'].sub(1).abs().idxmin(), 'year']}")

In [ ]:
year_cols = [c for c in df_clean.columns if c.isdigit()]

coverage = df_clean[year_cols].notna().sum().rename("non_null_count").to_frame()
coverage["coverage_pct"] = (coverage["non_null_count"] / len(df_clean) * 100).round(1)
coverage = coverage.sort_values("coverage_pct", ascending=False)

print("Data coverage by year:")
print(coverage.to_string())
print(f"\nBest year to anchor analysis: {coverage.index[0]} ({coverage['coverage_pct'].iloc[0]}% coverage)")

In [ ]:
male_age = df_clean[df_clean["Description"] == "Median age - males (years)"]
female_age = df_clean[df_clean["Description"] == "Median age - females (years)"]

year_cols = [c for c in df_clean.columns if c.isdigit()]

age_df = pd.DataFrame({
    "year": year_cols,
    "median_age_male": male_age[year_cols].values[0],
    "median_age_female": female_age[year_cols].values[0]
}).dropna()

age_df["age_gap_f_minus_m"] = (age_df["median_age_female"] - age_df["median_age_male"]).round(1)

print(age_df.to_string(index=False))
print(f"\nAverage gap (females older by): {age_df['age_gap_f_minus_m'].mean():.2f} years")
print(f"Trend: {'widening' if age_df['age_gap_f_minus_m'].iloc[-1] > age_df['age_gap_f_minus_m'].iloc[0] else 'narrowing'}")

In [ ]:
year_cols = [c for c in df_clean.columns if c.isdigit()]

category_stats = df_clean.groupby("Parent Description").agg(
    measure_count=("Description", "nunique"),
    avg_coverage_pct=(year_cols[0], lambda x: df_clean.loc[x.index, year_cols].notna().mean(axis=1).mean() * 100)
).round(1).sort_values("measure_count", ascending=False)

print("Top 10 parent categories by number of measures:")
print(category_stats.head(10).to_string())
print(f"\nTotal parent categories: {len(category_stats)}")
print(f"Most data-complete category: {category_stats['avg_coverage_pct'].idxmax()}")

In [138]:
import requests
import pandas as pd
import psycopg2
import geopandas as gpd
from psycopg2.extras import execute_values
from pathlib import Path
import time
import getpass
import sqlite3

# Task 2: Greater Sydney SA2/SA4 Points of Interest Dataset

## 2.1 Select SA4 Zone and Load SA2 Boundaries

Each group member should select one distinct Greater Sydney SA4 zone.

Selected zones:
- Ronnie: Sydney - Parramatta
- Spencer: Sydney - North Sydney and Hornsby
- Arya: Sydney - Baulkham Hills and Hawkesbury
- Ananya: Sydney - Blacktown

Each member's code is structured identically below. The SA2 digital boundary 
file is used to identify every SA2 region inside each SA4 and to get each 
SA2 bounding box for the POI API query.

In [ ]:
# --- Credentials (PostgreSQL — Ananya) ---
DB_CONFIG = {
    "host":     "localhost",
    "port":     5432,
    "dbname":   "postgres",
    "user":     input("PostgreSQL username: "),
    "password": input("PostgreSQL password (leave blank if none): "),
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

try:
    conn = get_connection()
    print("✓ Connected to PostgreSQL!")
    conn.close()
except Exception as e:
    print(f"✗ Connection failed: {e}")

# --- Shared paths ---
SA2_BOUNDARY_PATH = Path.cwd() / "SA2_2021_AUST_GDA2020.shp"
POI_CACHE_DIR = Path.cwd() / "poi_cache"
POI_CACHE_DIR.mkdir(exist_ok=True)

# --- Load all Greater Sydney SA2 boundaries ---
sa2_all = gpd.read_file(SA2_BOUNDARY_PATH).to_crs("EPSG:4326")
greater_sydney_sa2 = sa2_all[sa2_all["GCC_NAME21"] == "Greater Sydney"].copy()

def load_sa2_boundaries(sa4_name):
    """Filter SA2 boundaries to a given SA4 zone."""
    zone = greater_sydney_sa2[greater_sydney_sa2["SA4_NAME21"] == sa4_name].copy()
    boundaries = zone[["SA2_CODE21", "SA2_NAME21", "SA4_NAME21", "AREASQKM21", "geometry"]].copy()
    boundaries["bbox"] = boundaries.geometry.apply(lambda geom: geom.bounds)
    print(f"Selected SA4: {sa4_name}")
    print(f"Number of SA2 regions: {len(boundaries)}")
    display(boundaries[["SA2_CODE21", "SA2_NAME21", "SA4_NAME21", "AREASQKM21", "bbox"]].head())
    return boundaries

## 2.2 NSW Points of Interest API Function

Adapted from the Week 8 `nearbyPOI()` function. Instead of searching around 
a centre point, this function accepts explicit bounding box coordinates 
allowing precise querying per SA2 region. Pagination is included to handle 
SA2s with more than 2000 POIs.

In [139]:
NSW_POI_URL = "https://maps.six.nsw.gov.au/arcgis/rest/services/public/NSW_POI/MapServer/0/query"

def get_pois_in_bbox(min_lon, min_lat, max_lon, max_lat, page_size=2000, max_pages=10):
    """
    Returns all POIs from the NSW POI API within a specified bounding box.
    Adapted from the Week 8 nearbyPOI() function — accepts explicit bbox
    coordinates instead of a centre point, allowing precise querying per SA2.
    Includes pagination to handle regions with more than 2000 POIs.

    Parameters
    ----------
    min_lon, min_lat : float — bottom-left corner (longitude, latitude)
    max_lon, max_lat : float — top-right corner (longitude, latitude)
    page_size        : int  — records per page (API max is 2000)
    max_pages        : int  — safety cap to prevent infinite loops

    Returns
    -------
    pd.DataFrame of POI records, or empty DataFrame on failure.
    """
    all_rows = []
    offset = 0
    previous_offsets = set()

    for _ in range(max_pages):
        if offset in previous_offsets:
            break
        previous_offsets.add(offset)

        params = {
            "f":                 "json",
            "where":             "1=1",
            "outFields":         "*",
            "returnGeometry":    "true",
            "geometry":          f'{{"xmin":{min_lon},"ymin":{min_lat},"xmax":{max_lon},"ymax":{max_lat}}}',
            "geometryType":      "esriGeometryEnvelope",
            "inSR":              "4326",
            "outSR":             "4326",
            "spatialRel":        "esriSpatialRelIntersects",
            "resultRecordCount": page_size,
            "resultOffset":      offset,
        }

        try:
            response = requests.get(NSW_POI_URL, params=params, timeout=30)
            response.raise_for_status()
            payload = response.json()

            if "error" in payload:
                print(f"  API error: {payload['error']}")
                break

            features = payload.get("features", [])
            for feature in features:
                row = feature.get("attributes", {}).copy()
                geom = feature.get("geometry", {})
                row["longitude"] = geom.get("x")
                row["latitude"]  = geom.get("y")
                all_rows.append(row)

            if not payload.get("exceededTransferLimit") or not features:
                break

            offset += len(features)

        except requests.exceptions.RequestException as e:
            print(f"  Request failed: {e}")
            break

    return pd.DataFrame(all_rows)

# Quick test
test_df = get_pois_in_bbox(150.897, -33.779, 150.930, -33.748)
print(f"✓ Test POIs found: {len(test_df)}")
if not test_df.empty:
    print(test_df[["poiname", "poitype", "longitude", "latitude"]].head())

✓ Test POIs found: 143
             poiname poitype   longitude   latitude
0  BREWONGLE WALKWAY    Park  150.898434 -33.777748
1               None    Park  150.910845 -33.778514
2               None    Park  150.912078 -33.776707
3  JOSEPH FRANK PARK    Park  150.897448 -33.767396
4         ALPHA PARK    Park  150.904617 -33.771756


## 2.3 Loop Through SA2 Regions

Shared loop function used by all members. For each SA2 region:
1. Fetch all POIs within the SA2 bounding box from the API (with caching)
2. Use a spatial join to keep only POIs inside the actual SA2 polygon
3. Store results in a combined DataFrame

In [140]:
def fetch_pois_for_zone(sa2_boundaries):
    """
    Loops through all SA2 regions in a zone, fetches POIs,
    applies a spatial join, and returns a combined DataFrame.
    """
    poi_frames = []

    for index, (_, sa2_row) in enumerate(sa2_boundaries.iterrows(), start=1):
        sa2_name = sa2_row["SA2_NAME21"]
        sa2_code = sa2_row["SA2_CODE21"]
        cache_path = POI_CACHE_DIR / f"poi_{sa2_code}.csv"

        print(f"[{index}/{len(sa2_boundaries)}] {sa2_name} ...", end=" ")

        # Use cache if available
        if cache_path.exists():
            bbox_pois = pd.read_csv(cache_path)
            print(f"loaded {len(bbox_pois):,} POIs from cache")
        else:
            min_lon, min_lat, max_lon, max_lat = sa2_row.geometry.bounds
            bbox_pois = get_pois_in_bbox(min_lon, min_lat, max_lon, max_lat)
            bbox_pois.to_csv(cache_path, index=False)
            print(f"fetched {len(bbox_pois):,} POIs from API")

        if bbox_pois.empty:
            continue

        # Convert to GeoDataFrame
        points = gpd.GeoDataFrame(
            bbox_pois,
            geometry=gpd.points_from_xy(bbox_pois["longitude"], bbox_pois["latitude"]),
            crs="EPSG:4326",
        )

        # Spatial join — keep only POIs inside actual SA2 polygon
        sa2_polygon = gpd.GeoDataFrame(
            [sa2_row],
            geometry="geometry",
            crs=sa2_boundaries.crs
        )
        inside_sa2 = gpd.sjoin(
            points,
            sa2_polygon[["SA2_CODE21", "SA2_NAME21", "SA4_NAME21", "geometry"]],
            predicate="within",
            how="inner",
        )

        print(f"    → kept {len(inside_sa2):,} POIs inside SA2 polygon")

        if inside_sa2.empty:
            continue

        inside_sa2 = inside_sa2.drop(columns=["geometry", "index_right"], errors="ignore")
        poi_frames.append(pd.DataFrame(inside_sa2))
        time.sleep(0.5)

    result = pd.concat(poi_frames, ignore_index=True) if poi_frames else pd.DataFrame()
    return result


## 2.4 Store POI Dataset in Local Database
### Ronnie — Sydney - Parramatta (SQLite)

In [141]:
# --- Ronnie's SA2 boundaries ---
MY_SA4_RONNIE = "Sydney - Parramatta"
sa2_boundaries_ronnie = load_sa2_boundaries(MY_SA4_RONNIE)

# --- Ronnie's loop ---
pois_df_ronnie = fetch_pois_for_zone(sa2_boundaries_ronnie)
print(f"POI rows collected for {MY_SA4_RONNIE}: {len(pois_df_ronnie):,}")
display(pois_df_ronnie.head())

# --- Ronnie: store in SQLite ---
DATA_DIR = Path.cwd()
DB_PATH = DATA_DIR / "data2001_assignment.sqlite"

with sqlite3.connect(DB_PATH) as conn:
    if not pois_df_ronnie.empty:
        pois_df_ronnie.to_sql(
            "points_of_interest_parramatta",
            conn,
            if_exists="replace",
            index=False
        )
        print(f"✓ Saved {len(pois_df_ronnie):,} rows to points_of_interest_parramatta in {DB_PATH.name}")
    else:
        print("No POI data saved — check API query and SA2 boundary inputs.")

Selected SA4: Sydney - Parramatta
Number of SA2 regions: 34


,SA2_CODE21,SA2_NAME21,SA4_NAME21,AREASQKM21,bbox
544,125011475,Rookwood Cemetery,Sydney - Parramatta,3.0150,"(151.04630599248603, -33.88518105233612, 151.0..."
545,125011582,Auburn - Central,Sydney - Parramatta,3.7326,"(151.01393746888547, -33.86374540908072, 151.0..."
546,125011583,Auburn - North,Sydney - Parramatta,2.0966,"(151.019167481206, -33.855175380781695, 151.04..."
547,125011584,Auburn - South,Sydney - Parramatta,2.4313,"(151.01213246728338, -33.88133879918749, 151.0..."
548,125011585,Berala,Sydney - Parramatta,2.0865,"(151.0247795349445, -33.8855811709981, 151.042..."


[1/34] Rookwood Cemetery ... loaded 23 POIs from cache
    → kept 0 POIs inside SA2 polygon
[2/34] Auburn - Central ... loaded 101 POIs from cache
    → kept 0 POIs inside SA2 polygon
[3/34] Auburn - North ... loaded 60 POIs from cache
    → kept 0 POIs inside SA2 polygon
[4/34] Auburn - South ... loaded 52 POIs from cache
    → kept 0 POIs inside SA2 polygon
[5/34] Berala ... loaded 50 POIs from cache
    → kept 0 POIs inside SA2 polygon
[6/34] Lidcombe ... loaded 175 POIs from cache
    → kept 0 POIs inside SA2 polygon
[7/34] Regents Park ... loaded 44 POIs from cache
    → kept 0 POIs inside SA2 polygon
[8/34] Silverwater - Newington ... loaded 64 POIs from cache
    → kept 0 POIs inside SA2 polygon
[9/34] Wentworth Point - Sydney Olympic Park ... loaded 166 POIs from cache
    → kept 0 POIs inside SA2 polygon
[10/34] Ermington - Rydalmere ... loaded 170 POIs from cache
    → kept 0 POIs inside SA2 polygon
[11/34] Oatlands - Dundas Valley ... loaded 141 POIs from cache
    → kept 0 

""


No POI data saved — check API query and SA2 boundary inputs.


### Ananya — Sydney - Blacktown (PostgreSQL + PostGIS)

In [143]:
# --- Ananya's SA2 boundaries ---
MY_SA4_ANANYA = "Sydney - Blacktown"
sa2_boundaries_ananya = load_sa2_boundaries(MY_SA4_ANANYA)

# --- Ananya's loop ---
pois_df_ananya = fetch_pois_for_zone(sa2_boundaries_ananya)
print(f"POI rows collected for {MY_SA4_ANANYA}: {len(pois_df_ananya):,}")
display(pois_df_ananya.head())

# --- Create PostgreSQL + PostGIS table ---
def create_poi_table():
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute("CREATE EXTENSION IF NOT EXISTS postgis;")
            cur.execute("""
                CREATE TABLE IF NOT EXISTS points_of_interest (
                    id         SERIAL PRIMARY KEY,
                    sa2_code   TEXT,
                    sa2_name   TEXT,
                    sa4_name   TEXT,
                    poiname    TEXT,
                    poitype    TEXT,
                    poigroup   TEXT,
                    longitude  DOUBLE PRECISION,
                    latitude   DOUBLE PRECISION,
                    geom       GEOMETRY(Point, 4326),
                    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                );
            """)
            cur.execute("""
                CREATE INDEX IF NOT EXISTS idx_poi_geom
                ON points_of_interest USING GIST (geom);
            """)
        conn.commit()
    print("✓ Table 'points_of_interest' ready.")

create_poi_table()

# --- Ingest into PostgreSQL ---
def ingest_pois(df):
    if df.empty:
        return 0

    df.columns = [c.lower() for c in df.columns]
    count = 0

    with get_connection() as conn:
        with conn.cursor() as cur:
            for _, row in df.iterrows():
                cur.execute("""
                    INSERT INTO points_of_interest
                        (sa2_code, sa2_name, sa4_name, poiname, poitype,
                         poigroup, longitude, latitude, geom)
                    VALUES (%s, %s, %s, %s, %s, %s, %s, %s,
                            ST_SetSRID(ST_MakePoint(%s, %s), 4326))
                    ON CONFLICT DO NOTHING;
                """, (
                    row.get("sa2_code21"),
                    row.get("sa2_name21"),
                    row.get("sa4_name21"),
                    row.get("poiname"),
                    row.get("poitype"),
                    row.get("poigroup"),
                    row.get("longitude"),
                    row.get("latitude"),
                    row.get("longitude"),
                    row.get("latitude"),
                ))
                count += 1
        conn.commit()
    return count

n = ingest_pois(pois_df_ananya)
print(f"✓ {n:,} POIs ingested into PostgreSQL.")

# --- Verify ---
with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT sa2_name,
                   COUNT(*)                AS total_pois,
                   COUNT(DISTINCT poitype) AS unique_types
            FROM points_of_interest
            WHERE sa4_name = %s
            GROUP BY sa2_name
            ORDER BY total_pois DESC;
        """, (MY_SA4_ANANYA,))
        rows = cur.fetchall()
        cols = [desc[0] for desc in cur.description]

df_verify = pd.DataFrame(rows, columns=cols)
print(df_verify.to_string(index=False))

Selected SA4: Sydney - Blacktown
Number of SA2 regions: 24


,SA2_CODE21,SA2_NAME21,SA4_NAME21,AREASQKM21,bbox
319,116011303,Blacktown (East) - Kings Park,Sydney - Blacktown,7.9753,"(150.8930845128287, -33.78263766944216, 150.92..."
320,116011304,Blacktown (North) - Marayong,Sydney - Blacktown,7.3828,"(150.86585671591743, -33.76796701788305, 150.9..."
321,116011306,Doonside - Woodcroft,Sydney - Blacktown,9.9106,"(150.85509049481698, -33.79226694496207, 150.8..."
322,116011307,Lalor Park - Kings Langley,Sydney - Blacktown,11.7998,"(150.91634817777918, -33.7811572936743, 150.96..."
323,116011560,Blacktown - South,Sydney - Blacktown,3.3594,"(150.89690926389085, -33.80049582801894, 150.9..."


[1/24] Blacktown (East) - Kings Park ... loaded 196 POIs from cache
    → kept 103 POIs inside SA2 polygon
[2/24] Blacktown (North) - Marayong ... loaded 114 POIs from cache
    → kept 60 POIs inside SA2 polygon
[3/24] Doonside - Woodcroft ... loaded 114 POIs from cache
    → kept 81 POIs inside SA2 polygon
[4/24] Lalor Park - Kings Langley ... loaded 217 POIs from cache
    → kept 112 POIs inside SA2 polygon
[5/24] Blacktown - South ... loaded 58 POIs from cache
    → kept 23 POIs inside SA2 polygon
[6/24] Blacktown - West ... loaded 76 POIs from cache
    → kept 57 POIs inside SA2 polygon
[7/24] Seven Hills - Prospect ... loaded 191 POIs from cache
    → kept 83 POIs inside SA2 polygon
[8/24] Toongabbie - West ... loaded 69 POIs from cache
    → kept 31 POIs inside SA2 polygon
[9/24] Glenwood ... loaded 89 POIs from cache
    → kept 40 POIs inside SA2 polygon
[10/24] Acacia Gardens ... loaded 9 POIs from cache
    → kept 5 POIs inside SA2 polygon
[11/24] Quakers Hill ... loaded 78 PO

,objectid,topoid,poigroup,poitype,poiname,poilabel,poilabeltype,poialtlabel,poisourcefeatureoid,accesscontrol,...,centroidid,shapeuuid,changetype,processstate,urbanity,longitude,latitude,SA2_CODE21,SA2_NAME21,SA4_NAME21
0,1852,500246212,3,Park,NaN,Park,GENERIC,NaN,61,1,...,NaN,575c45e5-d50e-3cb3-bb8b-50d58a747176,I,NaN,U,150.912078,-33.776707,116011303,Blacktown (East) - Kings Park,Sydney - Blacktown
1,1856,500246289,3,Park,ALPHA PARK,ALPHA PARK,NAMED,NaN,61,1,...,NaN,432e1e3a-f451-336a-8a4e-bede68cecd1c,I,NaN,U,150.904617,-33.771756,116011303,Blacktown (East) - Kings Park,Sydney - Blacktown
2,1858,500246311,3,Park,BILLY GOAT HILL RESERVE,BILLY GOAT HILL RESERVE,NAMED,NaN,61,1,...,NaN,4bd4fd4e-3c72-3b72-b18b-f24d1ebcf9c4,I,NaN,U,150.916480,-33.766822,116011303,Blacktown (East) - Kings Park,Sydney - Blacktown
3,1863,500246389,3,Park,NaN,Park,GENERIC,NaN,61,1,...,NaN,2fcebdf1-043e-3a0e-b0e0-6b510d4faef7,I,NaN,U,150.917386,-33.754204,116011303,Blacktown (East) - Kings Park,Sydney - Blacktown
4,1864,500246411,3,Park,KINGSFORD RESERVE,KINGSFORD RESERVE,NAMED,NaN,61,1,...,NaN,598232ad-3981-3a9b-b412-3263d1c70139,I,NaN,U,150.924702,-33.751693,116011303,Blacktown (East) - Kings Park,Sydney - Blacktown


✓ Table 'points_of_interest' ready.
✓ 1,414 POIs ingested into PostgreSQL.
                     sa2_name  total_pois  unique_types
     Rooty Hill - Minchinbury         127            25
   Lalor Park - Kings Langley         112            21
        Mount Druitt - Whalan         110            26
Blacktown (East) - Kings Park         103            29
Bidwill - Hebersham - Emerton          93            17
       Seven Hills - Prospect          83            21
         Doonside - Woodcroft          81            19
                   Riverstone          69            31
    Lethbridge Park - Tregear          66            16
     Hassall Grove - Plumpton          63            18
 Blacktown (North) - Marayong          60            19
             Blacktown - West          57            14
                 Quakers Hill          54            16
           Prospect Reservoir          47            16
      Glendenning - Dean Park          42            14
                     Glenwood

In [142]:
with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS points_of_interest;")
    conn.commit()

### Spencer & Arya — Add your sections below following the same structure

# Task 3: SA2 Well-Resourced Score

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def z_score(series):
    std = series.std(ddof=0)
    if std == 0 or pd.isna(std):
        return pd.Series(0, index=series.index)
    return (series - series.mean()) / std

if not pois_df.empty and "sa2_name" in pois_df.columns:
    score_df = pois_df.groupby("sa2_name").size().rename("poi_count").reset_index()
    score_df["z_poi"] = z_score(score_df["poi_count"])
    score_df["score"] = sigmoid(score_df["z_poi"])
else:
    score_df = pd.DataFrame(columns=["sa2_name", "poi_count", "z_poi", "score"])

score_df.head()

## Scoring Explanation Notes

Use this section to explain:

- Why POI count is a reasonable proxy for resource availability
- Why z-score normalisation is used
- Why sigmoid scaling is used
- Whether population below 100 was excluded
- Any extensions to the formula, such as population adjustment or POI category weighting

# Task 4: Report Analysis and Visualisation

## 4.1 Key Findings from Task 1

Write the main statistical findings here after completing the derived statistics.

Possible angles:

- Population change over time
- Age structure
- Gender differences
- Density or growth indicators
- Measures with unusual changes or missingness

## 4.2 Score Visualisation Plan

Add plots here once `score_df` is populated.

Recommended visuals:

- Histogram of SA2 scores
- Top and bottom ranked SA2 regions
- Map overlay or choropleth if boundary geometry is available
- POI category breakdown by SA4 or SA2

In [ ]:
if not score_df.empty:
    display(score_df.sort_values("score", ascending=False).head(10))
    display(score_df.sort_values("score", ascending=True).head(10))
    display(score_df["score"].describe())
else:
    print("Score table is empty. Complete Task 2 before generating score summaries.")

## 4.3 Limitations

Discuss the limitations of the analysis here.

Possible limitations:

- POI count does not measure service quality or capacity
- Larger SA2s may naturally contain more POIs
- Population size may need to be considered
- API completeness and category definitions may affect results
- Bounding boxes can include POIs outside the actual SA2 polygon unless geometry filtering is added

# Next Steps

1. Add group member names and selected SA4 zones.
2. Complete Task 1 derived statistics.
3. Add SA2 boundary data for selected SA4 zones.
4. Confirm the NSW Points of Interest API endpoint from the Week 8 tutorial.
5. Store POI data in the local SQLite database.
6. Generate score visualisations and write report findings.